# Obnoxious _p_-Median Problem

## Problem definition

In the broader topic of facility location disparsing, obnoxious facilities were not taken into account when modelling such problems. Obnoxious facilities includes nuclear power plants, waste dumpsites and other structures that negatively impact clients, the environment and other facilities. The problem discussed in this notebook handles such cases, trying to maximize the distance between obnoxious plants and other facilities.

## The unified model
Based on the method of Ogryczak and Tamir for minimizing the sum of the K largest 
functions, Lei and Church proposed a unified dispersion model where distances are 
measured between facilities. The adaptation to the OpMP setting requires a key 
conceptual shift: distances are now measured from **clients to facilities** rather 
than between facilities. This change is reflected both in the index set (I instead 
of J for the outer variables) and in the coefficient of $s_i$, which becomes L instead 
of L+1, since client i is not itself a facility and contributes no zero self-distance.

### Parameters and Variables

**Sets:**
- $I = \{1, \dots, n\}$: set of clients
- $J = \{1, \dots, m\}$: set of potential facility sites

**Parameters:**
- $d(i, j)$: distance between client $i \in I$ and facility $j \in J$
- $p$: number of facilities to open
- $K$: number of clients considered in the objective (set to $n$ for the OpMP)
- $L$: number of closest facilities considered per client (set to $1$ for the OpMP)
- $M$: a sufficiently large constant

**Decision variables:**
- $y_j \in \{0,1\}$: equals $1$ if facility $j$ is opened, $0$ otherwise

**Auxiliary variables:**
- $t \in \mathbb{R}$: threshold value used to identify the K smallest partial sums
- $u_i \geq 0$: excess of client $i$'s contribution over the threshold $t$
- $q_i \geq 0$: intermediate variable representing the weighted sum of distances 
  for client $i$, computed from $s_i$ and $z(i,j)$
- $s_i \geq 0$: plays a role analogous to $t$, but at the client level
- $z(i,j) \geq 0$: non-zero when facility $j$ is among the $L$ closest to client $i$

### Model Formulation

$$\max \; Kt - \sum_{i \in I} u_i \tag{12}$$

**Subject to:**

$$t - u_i \leq q_i \qquad \forall i \in I \tag{13}$$

$$q_i = Ls_i - \sum_{j \in J} z(i,j) \qquad \forall i \in I \tag{14}$$

$$s_i - z(i,j) \leq d(i,j) + M(1 - y_j) \qquad \forall i \in I, \forall j \in J \tag{15}$$

$$\sum_{j \in J} y_j = p \tag{3}$$

$$u_i, z(i,j) \geq 0, \quad y_j \in \{0,1\} \qquad \forall i \in I, \forall j \in J$$

**Intuition behind each constraint:**

- **(12)** — The objective maximizes the sum of the $K$ smallest client-to-facility 
  distances. With $K=n$ this reduces to maximizing the sum over all clients, 
  which is exactly the OpMP objective.

- **(13)** — Links $u_i$ to the threshold $t$: if client $i$ is among the $K$ 
  worst-served, $u_i = 0$ and the full contribution $q_i$ is counted; 
  otherwise $u_i$ absorbs the excess.

- **(14)** — Defines $q_i$ as the weighted minimum distance from client $i$ 
  to the $L$ nearest open facilities. With $L=1$ this simplifies to the 
  distance from client $i$ to its single nearest open facility.

- **(15)** — The big-M constraint: if facility $j$ is open ($y_j = 1$), 
  forces $s_i - z(i,j) \leq d(i,j)$, effectively capturing the distance 
  from client $i$ to facility $j$ in the auxiliary variables.

- **(3)** — Exactly $p$ facilities must be opened.